In [25]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [26]:
class SimpleLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super(SimpleLSTM, self).__init__()

        self.lstm = nn.LSTM(input_size=input_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            dropout=dropout)

        self.bn = nn.BatchNorm1d(hidden_size)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):  # stock_ids can be used later
        lstm_out, _ = self.lstm(x)                      # [B, T, H]
        out = lstm_out[:, -1, :]                        # Take last timestep
        # out = self.bn(out)                              # BatchNorm over features
        out = self.fc(out)                              # [B, 1]
        return out
        
class StockDataset(Dataset):
    def __init__(self, sequences, targets, stock_ids):
        self.sequences = sequences
        self.targets = targets
        self.stock_ids = stock_ids
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return (torch.FloatTensor(self.sequences[idx]),
                torch.FloatTensor([self.targets[idx]]),
                torch.LongTensor([self.stock_ids[idx]]))

In [27]:
def prepare_stock_data(df, seq_len=20, train_split=0.7, val_split=0.15):
    """Fixed data preparation - NO SCALING"""
    stocks = sorted(df['Ticker'].unique())
    stock_to_id = {stock: i for i, stock in enumerate(stocks)}

    # feature_cols = [feature for feature in df.columns if feature not in ['Date', 'Ticker', 'Target']]
    feature_cols = [
        'Open', 'High', 'Low', 'Close', 'Volume'
    ]

    all_sequences = []
    all_targets = []
    all_stock_ids = []
    
    for stock in stocks:
        stock_data = df[df['Ticker'] == stock].copy()
        stock_data = stock_data.sort_values('Date').reset_index(drop=True)
        
        if len(stock_data) < seq_len + 1:
            continue
        
        X = stock_data[feature_cols].values
        y = stock_data['Target'].values

        for i in range(seq_len, len(X)):
            all_sequences.append(X[i-seq_len:i])
            all_targets.append(y[i])
            all_stock_ids.append(stock_to_id[stock])
    
    X_seq = np.array(all_sequences)
    y_seq = np.array(all_targets)
    stock_ids = np.array(all_stock_ids)
    
    # Split data
    train_idx = int(len(X_seq) * train_split)
    val_idx = int(len(X_seq) * (train_split + val_split))
    
    return ((X_seq[:train_idx], y_seq[:train_idx], stock_ids[:train_idx]), 
            (X_seq[train_idx:val_idx], y_seq[train_idx:val_idx], stock_ids[train_idx:val_idx]), 
            (X_seq[val_idx:], y_seq[val_idx:], stock_ids[val_idx:]), 
            stock_to_id)

# Load and prepare data
df = pd.read_csv("../data/processed/stock_data_optimized.csv")
(X_train, y_train, stock_train), (X_val, y_val, stock_val), (X_test, y_test, stock_test), stock_to_id = prepare_stock_data(df)

print(f"\n✅ Tickers used: {list(stock_to_id.keys())}")

print(f"Data: {len(X_train)} train, {len(X_val)} val, {len(X_test)} test")
print(f"Features: {X_train.shape[2]}")
print(f"Class distribution: {np.bincount(y_train.astype(int))}")
print(f"Train target mean: {y_train.mean():.3f}")
print(f"Val target mean: {y_val.mean():.3f}")

print(f"\nSequence alignment check:")
print(f"Last feature value: {X_train[0, -1, 0]:.3f}")  # Last timestep of first sequence
print(f"Target: {y_train[0]}")

# Check data statistics
print(f"\nData statistics:")
print(f"X_train stats - mean: {X_train.mean():.3f}, std: {X_train.std():.3f}")
print(f"X_train range - min: {X_train.min():.3f}, max: {X_train.max():.3f}")



✅ Tickers used: ['AAPL', 'AMZN', 'GOOGL', 'MSFT', 'NVDA']
Data: 9884 train, 2118 val, 2118 test
Features: 5
Class distribution: [4400 5484]
Train target mean: 0.555
Val target mean: 0.576

Sequence alignment check:
Last feature value: 4.292
Target: 0.0

Data statistics:
X_train stats - mean: -0.099, std: 0.979
X_train range - min: -1.579, max: 12.845


In [28]:
print(df.describe())
# show df head
print("\nData sample:")
print(df.head())

# show df tail
print("\nData sample:")
print(df.tail())

               Open          High           Low         Close        Volume  \
count  1.422000e+04  1.422000e+04  14220.000000  1.422000e+04  14220.000000   
mean  -3.197942e-17 -7.994855e-17      0.000000 -7.994855e-17      0.000000   
std    1.000035e+00  1.000035e+00      1.000035  1.000035e+00      1.000035   
min   -1.157369e+00 -1.156427e+00     -1.160000 -1.160339e+00     -1.598635   
25%   -7.963815e-01 -7.975978e-01     -0.797556 -7.970500e-01     -0.553098   
50%   -2.537059e-01 -2.539440e-01     -0.256775 -2.551789e-01     -0.425427   
75%    4.718529e-01  4.739719e-01      0.470068  4.713671e-01      0.236814   
max    5.740759e+00  5.676885e+00      5.740353  5.690981e+00     12.844596   

         Log_Return  High_Low_Ratio  Close_Open_Ratio           Gap  \
count  1.422000e+04    1.422000e+04      1.422000e+04  1.422000e+04   
mean  -6.995498e-18   -1.886786e-15      2.620314e-15 -2.998071e-18   
std    1.000035e+00    1.000035e+00      1.000035e+00  1.000035e+00   
min 

In [29]:

X_train_all = X_train
X_val_all = X_val
X_test_all = X_test

train_dataset = StockDataset(X_train_all, y_train, stock_train)
val_dataset = StockDataset(X_val_all, y_val, stock_val)
test_dataset = StockDataset(X_test_all, y_test, stock_test)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleLSTM(input_size=X_train_all.shape[2]).to(device)

criterion = nn.BCEWithLogitsLoss()  # No pos_weight
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {device}")


Model parameters: 201,601
Device: cpu


In [30]:
train_losses = []
val_losses = []
train_accs = []
val_accs = []
learning_rates = []

num_epochs = 200
best_val_loss = float('inf')
patience_counter = 0
patience = 50

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False) 
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    # Debug: track predictions
    batch_predictions_mean = []
    
    for batch_idx, (batch_X, batch_y, batch_stock_ids) in enumerate(train_loader):
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X).squeeze()
        loss = criterion(outputs, batch_y.squeeze())
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        train_loss += loss.item()
        predictions = (torch.sigmoid(outputs) > 0.5).float()
        
        batch_predictions_mean.append(predictions.mean().item())
        train_correct += (predictions == batch_y.squeeze()).sum().item()
        train_total += len(batch_y)
    
    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for batch_X, batch_y, batch_stock_ids in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            outputs = model(batch_X).squeeze()
            loss = criterion(outputs, batch_y.squeeze())
            val_loss += loss.item()
            
            predictions = (torch.sigmoid(outputs) > 0.5).float()
            val_correct += (predictions == batch_y.squeeze()).sum().item()
            val_total += len(batch_y)
    
    # Calculate metrics
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    current_lr = optimizer.param_groups[0]['lr']
    
    # Store metrics
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    learning_rates.append(current_lr)
    
    # Print progress
    if epoch % 5 == 0 or epoch < 200:
        print(f"Epoch {epoch+1:3d}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}, "
              f"Train Acc={train_acc:.3f}, Val Acc={val_acc:.3f}, LR={current_lr:.6f}")
    
    # Scheduler step
    scheduler.step(avg_val_loss)
    
    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

print(f"\nBest validation loss: {best_val_loss:.4f}")

# Test predictions distribution
model.eval()
test_predictions = []
with torch.no_grad():
    for batch_X, batch_y, batch_stock_ids in test_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X).squeeze()
        probs = torch.sigmoid(outputs)
        test_predictions.extend(probs.cpu().numpy())

print(f"\nTest predictions - mean: {np.mean(test_predictions):.3f}, std: {np.std(test_predictions):.3f}")
print(f"Min: {np.min(test_predictions):.3f}, Max: {np.max(test_predictions):.3f}")

Epoch   1: Train Loss=0.6883, Val Loss=0.6823, Train Acc=0.553, Val Acc=0.576, LR=0.001000
Epoch   2: Train Loss=0.6872, Val Loss=0.6836, Train Acc=0.555, Val Acc=0.576, LR=0.001000
Epoch   3: Train Loss=0.6870, Val Loss=0.6845, Train Acc=0.554, Val Acc=0.576, LR=0.001000
Epoch   4: Train Loss=0.6871, Val Loss=0.6824, Train Acc=0.554, Val Acc=0.576, LR=0.001000
Epoch   5: Train Loss=0.6866, Val Loss=0.6831, Train Acc=0.555, Val Acc=0.577, LR=0.001000
Epoch   6: Train Loss=0.6868, Val Loss=0.6827, Train Acc=0.556, Val Acc=0.578, LR=0.001000
Epoch   7: Train Loss=0.6866, Val Loss=0.6818, Train Acc=0.555, Val Acc=0.579, LR=0.001000
Epoch   8: Train Loss=0.6863, Val Loss=0.6806, Train Acc=0.556, Val Acc=0.575, LR=0.001000
Epoch   9: Train Loss=0.6866, Val Loss=0.6809, Train Acc=0.555, Val Acc=0.576, LR=0.001000
Epoch  10: Train Loss=0.6866, Val Loss=0.6838, Train Acc=0.556, Val Acc=0.571, LR=0.001000
Epoch  11: Train Loss=0.6866, Val Loss=0.6807, Train Acc=0.553, Val Acc=0.580, LR=0.001000

KeyboardInterrupt: 

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Load your processed data
df = pd.read_csv("../data/processed/stock_data_optimized.csv")

# Select features and target (drop Date, Ticker, Target)
exclude = ['Date', 'Ticker', 'Target']
# features = [col for col in df.columns if col not in exclude]
# add ohlcv features
features = [
    'Open', 'High', 'Low', 'Close', 'Volume'
]
X = df[features]
y = df['Target'].astype(int)

# (Recommended!) Do a time-based split if possible; else, use random split:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, shuffle=False  # shuffle=False for time order
)

# Logistic regression (with regularization)
clf = LogisticRegression(
    penalty='l2',    # Regularization
    C=1.0,           # Inverse regularization strength (lower = stronger)
    max_iter=200,
    solver='lbfgs'
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_val)

# Evaluate
acc = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {acc:.4f}")
print("Classification Report:\n", classification_report(y_val, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))


In [ ]:
# Load best model and evaluate
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

all_predictions = []
all_targets = []
all_probs = []

with torch.no_grad():
    for batch_X, batch_y, batch_stock_ids in test_loader:
        outputs = model(batch_X, batch_stock_ids.squeeze()).squeeze()
        probs = torch.sigmoid(outputs)
        predictions = (probs > 0.5).float()
        
        all_predictions.extend(predictions.cpu().numpy())
        all_targets.extend(batch_y.squeeze().cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Calculate metrics
accuracy = accuracy_score(all_targets, all_predictions)
print(f"Final Test Accuracy: {accuracy:.3f}")

# Create comprehensive visualizations
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. Training curves
epochs = range(1, len(train_losses) + 1)
axes[0,0].plot(epochs, train_losses, 'b-', label='Train')
axes[0,0].plot(epochs, val_losses, 'r-', label='Validation')
axes[0,0].set_title('Loss Curves')
axes[0,0].legend()
axes[0,0].grid(True)

# 2. Accuracy curves
axes[0,1].plot(epochs, train_accs, 'b-', label='Train')
axes[0,1].plot(epochs, val_accs, 'r-', label='Validation')
axes[0,1].set_title('Accuracy Curves')
axes[0,1].legend()
axes[0,1].grid(True)

# 3. Learning rate
axes[0,2].plot(epochs, learning_rates, 'g-')
axes[0,2].set_title('Learning Rate')
axes[0,2].set_yscale('log')
axes[0,2].grid(True)

# 4. Confusion matrix
cm = confusion_matrix(all_targets, all_predictions)
sns.heatmap(cm, annot=True, fmt='d', ax=axes[1,0], 
           xticklabels=['Down', 'Up'], yticklabels=['Down', 'Up'])
axes[1,0].set_title('Confusion Matrix')

# 5. Prediction distribution
axes[1,1].hist(all_probs, bins=50, alpha=0.7, edgecolor='black')
axes[1,1].axvline(x=0.5, color='r', linestyle='--', label='Threshold')
axes[1,1].set_title('Prediction Probability Distribution')
axes[1,1].legend()

# 6. ROC-like accuracy by threshold
thresholds = np.linspace(0, 1, 101)
threshold_accs = []
for thresh in thresholds:
    preds = (np.array(all_probs) > thresh).astype(int)
    acc = accuracy_score(all_targets, preds)
    threshold_accs.append(acc)

axes[1,2].plot(thresholds, threshold_accs)
axes[1,2].axvline(x=0.5, color='r', linestyle='--', label='Current threshold')
axes[1,2].set_title('Accuracy vs Threshold')
axes[1,2].set_xlabel('Threshold')
axes[1,2].set_ylabel('Accuracy')
axes[1,2].legend()
axes[1,2].grid(True)

plt.tight_layout()
plt.show()

# Classification report
print("\nDetailed Classification Report:")
print(classification_report(all_targets, all_predictions, target_names=['Down', 'Up']))